# 🏆 Notebook 04 — Next Month Top Category Classifier

**Model**: `top_category_classifier.pkl`  
**Algorithm**: Random Forest Classifier  
**Target**: Department with highest sales next month (multi-class classification)  
**Input Features**: Monthly sales per department vector + growth rates

In [ ]:
# What: Import classification libraries and label encoding tools
# Why: Multi-class classification (21 departments) needs RF Classifier + LabelEncoder
# Method: sklearn RandomForestClassifier + LabelEncoder + standard data stack
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

DATA_PATH    = '../data'
MODELS_PATH  = '../models'
RANDOM_STATE = 42
os.makedirs(MODELS_PATH, exist_ok=True)

# Department avg price proxy (Rs.) — consistent with all other notebooks
DEPT_PRICE = {1:350,2:200,3:250,4:150,5:800,6:400,7:180,8:600,9:300,
              10:200,11:500,12:700,13:350,14:280,15:200,16:300,17:450,
              18:500,19:220,20:400,21:200}
print('Config loaded ✅')

In [ ]:
# What: Load all required dataset files
# Why: Need product-department links + order data to compute per-department monthly sales
# Method: Read four CSV files and verify shapes
orders      = pd.read_csv(f'{DATA_PATH}/orders.csv')
op_train    = pd.read_csv(f'{DATA_PATH}/order_products__train.csv')
products    = pd.read_csv(f'{DATA_PATH}/products.csv')
departments = pd.read_csv(f'{DATA_PATH}/departments.csv')

print(f'orders: {orders.shape} | op_train: {op_train.shape}')
print(f'products: {products.shape} | departments: {departments.shape}')

In [ ]:
# What: Build enriched item table with department and revenue columns
# Why: Need department_id per item to count sales per department per month
# Method: Merge op_train → products (get department_id) → orders (get order_number)
products_dept = products.merge(departments, on='department_id', how='left')
order_items = op_train.merge(products_dept[['product_id','department_id']], on='product_id', how='left')
order_items = order_items.merge(
    orders[['order_id','user_id','order_number']],
    on='order_id', how='left'
)

order_items['price']    = order_items['department_id'].map(DEPT_PRICE).fillna(200)
order_items['revenue']  = order_items['price']
order_items['month_num'] = np.ceil(order_items['order_number'] / 3).astype(int)

print('Item table with departments:')
print(order_items[['order_id','department_id','revenue','month_num']].head())

In [ ]:
# What: Compute monthly revenue per department (pivot table)
# Why: Features for classifier = sales volume per department per month
# Method: groupby (month_num, department_id) → sum revenue → pivot to wide format
month_dept = order_items.groupby(['month_num','department_id'])['revenue'].sum().reset_index()
month_dept.columns = ['month_num','department_id','dept_revenue']

# Pivot — each row = one month; each column = one department's revenue
pivot = month_dept.pivot_table(index='month_num', columns='department_id',
                                values='dept_revenue', fill_value=0)
pivot.columns = [f'dept_{c}' for c in pivot.columns]  # rename for clarity
pivot.reset_index(inplace=True)

print(f'Pivot table shape: {pivot.shape}')
print(pivot.head(3).to_string())

In [ ]:
# What: Identify the top department (by revenue) in each month
# Why: This becomes the classification TARGET — which department wins next month?
# Method: argmax across department revenue columns → department id → label
dept_cols = [c for c in pivot.columns if c.startswith('dept_')]

# Get department with max revenue per month
pivot['top_dept_col'] = pivot[dept_cols].idxmax(axis=1)
pivot['top_dept_id']  = pivot['top_dept_col'].str.replace('dept_','').astype(int)

# Merge with department name for readability
pivot = pivot.merge(departments, left_on='top_dept_id', right_on='department_id', how='left')

print('Top department per month:')
print(pivot[['month_num','top_dept_id','department']].head(15).to_string(index=False))
print('\nClass distribution:')
print(pivot['department'].value_counts())

In [ ]:
# What: Add lag features — previous month's dept sales as predictors
# Why: This month's department dominance predicts next month's top category
# Method: shift(1) each department revenue column to create lag features
for col in dept_cols:
    pivot[f'prev_{col}'] = pivot[col].shift(1)
    pivot[f'growth_{col}'] = pivot[col].pct_change()  # month-over-month growth per dept

pivot.dropna(inplace=True)  # remove first row (no lag)
print(f'After lag engineering: {pivot.shape}')

In [ ]:
# What: Encode target labels as integers for classifier
# Why: sklearn classifiers require numeric labels, not string department names
# Method: LabelEncoder converts department names to 0-based integers
le = LabelEncoder()
pivot['target_label'] = le.fit_transform(pivot['top_dept_id'])

print('Label mapping (first 10):')
for cls, idx in zip(le.classes_[:10], range(10)):
    print(f'  {idx} → dept_id {cls}')

In [ ]:
# What: Prepare feature matrix X and label vector y
# Why: Classifier needs current + lagged department sales as input features
# Method: Use current + lag dept revenue columns; exclude metadata
feature_cols = dept_cols + [f'prev_{c}' for c in dept_cols] + [f'growth_{c}' for c in dept_cols]

X = pivot[feature_cols].values
y = pivot['target_label'].values

# Stratified split to preserve class proportions in train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y if len(np.unique(y)) > 1 else None
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Unique classes: {len(np.unique(y))}')

In [ ]:
# What: Train Random Forest Classifier for top category prediction
# Why: RF handles high-dimensional (multi-dept) features and is robust for classification
# Method: 500 trees, balanced class weights for any imbalanced month distribution
model = RandomForestClassifier(
    n_estimators    = 500,
    class_weight    = 'balanced',   # handle any class imbalance
    max_features    = 'sqrt',       # random feature subsets per split
    min_samples_leaf = 2,
    random_state    = RANDOM_STATE,
    n_jobs          = -1
)
model.fit(X_train, y_train)
print('Training complete ✅')

In [ ]:
# What: Evaluate classifier accuracy and print classification report
# Why: Measure how accurately model identifies the top revenue department
# Method: accuracy_score + full classification_report with precision/recall/F1
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Top-Category Classifier Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print()
print('Classification Report:')
target_names = [str(departments[departments['department_id']==int(le.inverse_transform([c])[0])]['department'].values[0])
                if int(le.inverse_transform([c])[0]) in departments['department_id'].values else str(c)
                for c in sorted(np.unique(y_test))]
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# What: Plot top-10 most important features for classification
# Why: Understand which departments' sales volume drive the top category prediction
# Method: Sort feature importances, plot top 10 as horizontal bar chart
fi = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 6))
fi.sort_values().plot(kind='barh', color='darkorange', edgecolor='white')
plt.title('Top 15 Feature Importances — Top Category Classifier', fontsize=13, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig(f'{MODELS_PATH}/chart_top_category_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance chart saved ✅')

In [ ]:
# What: Save the classifier AND label encoder together
# Why: Inference needs both model (to predict label int) and encoder (to decode back to dept name)
# Method: Save as dict with both objects — single .pkl file for clean API usage
classifier_bundle = {
    'model':         model,
    'label_encoder': le,
    'feature_cols':  feature_cols,
    'dept_map':      dict(zip(departments['department_id'], departments['department']))
}
joblib.dump(classifier_bundle, f'{MODELS_PATH}/top_category_classifier.pkl')
print('✅ Saved: models/top_category_classifier.pkl')
print(f'   Size: {os.path.getsize(f"{MODELS_PATH}/top_category_classifier.pkl") / 1024:.1f} KB')

In [ ]:
# What: Demonstrate inference — predict top category for next month
# Why: Confirm pkl bundle loads and decodes prediction to department name
# Method: Load bundle, predict on last month features, decode label to name
bundle = joblib.load(f'{MODELS_PATH}/top_category_classifier.pkl')

last_features = pivot[feature_cols].iloc[[-1]].values
pred_label    = bundle['model'].predict(last_features)[0]
pred_dept_id  = int(bundle['label_encoder'].inverse_transform([pred_label])[0])
pred_name     = bundle['dept_map'].get(pred_dept_id, 'Unknown')

# Probability for top class
prob = bundle['model'].predict_proba(last_features).max()

print('=== Sample Prediction (Next Month Top Category) ===')
print(f'🏆 Top Category    : {pred_name.title()}')
print(f'   Department ID   : {pred_dept_id}')
print(f'   Confidence      : {prob*100:.1f}%')